# SpectralGuard Phase 5 — availability under adversarial-benign traffic
Select an H100 runtime and click **Run all** once. This fifth and final experiment notebook freezes the Phase 3 detector, constructs held-out multi-turn sessions, selects Algorithm 2b only on validation, evaluates the test set once, performs the CPU-safe analysis, and creates `phase5_experiments.zip` plus `phase5_final.zip`. If the 2.7-hour reserve is reached it creates `phase5_handoff.tar.gz` and resumes without repeating completed work.

In [ ]:
from pathlib import Path
import hashlib
import os
import subprocess
import sys
if not Path('configs/phase5.yaml').is_file():
    os.chdir('..')
assert Path('configs/phase5.yaml').is_file(), f'Repository root not found from {Path.cwd()}'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-lightning.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)
subprocess.run([sys.executable, 'scripts/install_lightning_mamba_kernels.py', '--cuda-smoke'], check=True)
print('Finished')

In [ ]:
import torch
from src.experiments.phase5_common import phase5_preflight
assert torch.cuda.is_available() and 'H100' in torch.cuda.get_device_name(0).upper()
phase5_preflight(require_h100=True)
print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0)})
print('Finished')

In [ ]:
from src.experiments.phase5_runner import run_automatic_phase5
phase5_report = run_automatic_phase5()
assert phase5_report['status'] in {'COMPLETE', 'TIMEBOX_HANDOFF'}
print(phase5_report)
print('Finished')

In [ ]:
complete = phase5_report['status'] == 'COMPLETE'
result = Path('Experiments/exports/phase5_experiments.zip' if complete else 'Experiments/exports/phase5_handoff.tar.gz').resolve()
assert result.is_file() and result.stat().st_size > 0
digest = hashlib.sha256(result.read_bytes()).hexdigest()
print(f'DOWNLOAD READY: {result}')
print(f'SHA256: {digest}')
if complete:
    final = Path('Experiments/exports/phase5_final.zip').resolve()
    assert final.is_file() and final.stat().st_size > 0
    print(f'DOWNLOAD READY: {final}')
    print(f'SHA256: {hashlib.sha256(final.read_bytes()).hexdigest()}')
print('STOP THE H100 NOW.' if complete else 'STOP THE H100 NOW. Upload this handoff to the next session.')
print('Finished')